amazon Replacement

In [ ]:
import pandas as pd
import os
import random
import torch

# Set random seed
random.seed(2024)

# ==================== Configuration Parameters ====================
input_path = 'C:/Users/THINK BOOK-16/Desktop/sport/'  # Directory containing inter.csv
output_path = 'C:/Users/THINK BOOK-16/Desktop/sport-processed-10/'
dataset_name = 'toy'

user_threshold = 5
item_threshold = 5
max_seq_len = 50

# ==================== Generate only 10% noisy version ====================
NOISE_RATIO = 0.10  # Generate only 10% noise

# ==================== Helper Functions ====================
def add_noise_guarantee_threshold(dataset, noise_ratio=0.05):
    """
    Add noise while ensuring:
    1. All items have interaction count ≥ item_threshold
    2. Item count remains unchanged (no new items introduced, no items lost)
    3. Only modify item_id
    4. Precisely control total noise ratio to noise_ratio
    """
    # Get clean data information
    clean_items = dataset['item_id'].unique().tolist()
    clean_item_counts = dataset['item_id'].value_counts().to_dict()
    
    print(f"  Clean item count: {len(clean_items)}")
    print(f"  Clean minimum interaction count: {min(clean_item_counts.values())}")
    
    # Copy dataset
    noisy_dataset = dataset.copy()
    
    # Calculate total interactions to modify
    total_interactions = len(noisy_dataset)
    target_changes = int(total_interactions * noise_ratio)
    print(f"  Total interactions: {total_interactions}")
    print(f"  Target modifications: {target_changes} (expected noise ratio: {noise_ratio*100:.1f}%)")
    
    # Track current interaction counts for each item (dynamically updated)
    current_item_counts = clean_item_counts.copy()
    
    # Phase 1: Collect all possible modifyable interactions
    print("  Phase 1: Collecting candidate modification positions...")
    candidate_indices = []
    
    for idx in noisy_dataset.index:
        old_item = noisy_dataset.at[idx, 'item_id']
        # Only consider items that would still meet threshold after decreasing by 1
        if current_item_counts[old_item] > item_threshold:
            candidate_indices.append(idx)
    
    print(f"  Found {len(candidate_indices)} candidate modification positions")
    
    # Randomly shuffle candidate positions
    random.shuffle(candidate_indices)
    
    # Phase 2: Perform modifications until target is reached
    print("  Phase 2: Executing modifications...")
    total_changes = 0
    max_attempts_per_change = 50
    
    for idx in candidate_indices:
        if total_changes >= target_changes:
            break
            
        old_item = noisy_dataset.at[idx, 'item_id']
        
        # Try to find a suitable replacement item
        attempts = 0
        new_item = None
        
        while attempts < max_attempts_per_change and new_item is None:
            # Randomly select a different item
            candidate = random.choice([item for item in clean_items if item != old_item])
            
            # Check if replacement is safe
            if (current_item_counts[old_item] - 1 >= item_threshold and 
                current_item_counts[candidate] + 1 >= item_threshold):
                new_item = candidate
                break
            attempts += 1
        
        if new_item is None:
            # Cannot find suitable replacement, skip
            continue
        
        # Perform replacement
        noisy_dataset.at[idx, 'item_id'] = new_item
        
        # Update counts
        current_item_counts[old_item] -= 1
        current_item_counts[new_item] += 1
        
        total_changes += 1
        
        # Show progress
        if total_changes % 500 == 0 or total_changes == target_changes:
            progress = total_changes / target_changes * 100
            print(f"    Progress: {total_changes}/{target_changes} ({progress:.1f}%)")
    
    # Calculate actual noise ratio
    actual_changes = 0
    for idx in noisy_dataset.index:
        # Find corresponding original index
        if idx in dataset.index:
            if dataset.at[idx, 'item_id'] != noisy_dataset.at[idx, 'item_id']:
                actual_changes += 1
    
    actual_noise_ratio = actual_changes / total_interactions
    
    print(f"\n  Noise addition completed:")
    print(f"    Target modifications: {target_changes}")
    print(f"    Actual modifications: {total_changes}")
    print(f"    Actual noise ratio: {actual_noise_ratio*100:.2f}%")
    
    # If actual ratio deviates significantly from target, show warning
    if abs(actual_noise_ratio - noise_ratio) > 0.005:  # 0.5% tolerance
        print(f"  Notice: Actual noise ratio deviates from target")
        print(f"    Target: {noise_ratio*100:.2f}%")
        print(f"    Actual: {actual_noise_ratio*100:.2f}%")
    
    # Verify item ID set
    noisy_items = set(noisy_dataset['item_id'].unique())
    clean_items_set = set(clean_items)
    
    if noisy_items == clean_items_set:
        print(f"  Item ID sets are identical")
    else:
        print(f"  Item ID sets do not match!")
        print(f"    Clean unique: {clean_items_set - noisy_items}")
        print(f"    Noisy unique: {noisy_items - clean_items_set}")
        return None, 0, 0
    
    # Verify interaction counts
    item_counts = noisy_dataset['item_id'].value_counts()
    min_interactions = item_counts.min()
    
    if min_interactions >= item_threshold:
        print(f"  All items have interaction count ≥ {item_threshold}")
        print(f"    Minimum interaction count: {min_interactions}")
    else:
        print(f"  Some items have interaction count < {item_threshold}")
        below_threshold = item_counts[item_counts < item_threshold]
        print(f"    Count: {len(below_threshold)}")
        print(f"    Item IDs: {list(below_threshold.index)}")
        return None, 0, 0
    
    # Verify total interaction count unchanged
    if len(noisy_dataset) == len(dataset):
        print(f"  Total interaction count unchanged: {len(noisy_dataset)}")
    else:
        print(f"  Total interaction count changed: {len(dataset)} -> {len(noisy_dataset)}")
        return None, 0, 0
    
    return noisy_dataset, total_changes, total_interactions

def truncate_or_pad(seq):
    """Truncate or pad sequence to fixed length (maintain original logic)"""
    cur_seq_len = len(seq)
    if cur_seq_len > max_seq_len:
        return seq[-max_seq_len:], max_seq_len
    else:
        PAD = 0
        return seq + [PAD] * (max_seq_len - cur_seq_len), cur_seq_len

# ==================== Main Process ====================

print("="*60)
print("Starting Data Processing (10% noise addition, precise noise ratio control)")
print("="*60)

# 1. Load inter.csv file
print("\n1. Loading inter.csv file...")
inter_file = os.path.join(input_path, 'inter.csv')

if not os.path.exists(inter_file):
    print(f"Error: Cannot find file {inter_file}")
    exit(1)

# Read inter.csv file
dataset = pd.read_csv(inter_file)

print(f"Original data statistics:")
print(f"  Data shape: {dataset.shape}")
print(f"  Number of users: {dataset['user_id'].nunique()}")
print(f"  Number of items: {dataset['item_id'].nunique()}")
print(f"  Number of interactions: {len(dataset)}")

# Check if domain column exists, if not add one
if 'domain' not in dataset.columns:
    print("Note: No domain column in inter.csv, adding default domain=0")
    dataset['domain'] = 0

# 2. Filter dataset (based on interaction frequency)
print("\n2. Filtering dataset...")
filtered_dataset = dataset.copy()
while True:
    ori_len = len(filtered_dataset)
    
    # Filter users with less than user_threshold interactions
    filtered_dataset = filtered_dataset[filtered_dataset['user_id'].map(filtered_dataset['user_id'].value_counts()) >= user_threshold]
    # Filter items with less than item_threshold interactions
    filtered_dataset = filtered_dataset[filtered_dataset['item_id'].map(filtered_dataset['item_id'].value_counts()) >= item_threshold]
    
    if len(filtered_dataset) == ori_len:
        break

print(f"\nFiltered data statistics:")
print(f"  Number of users: {filtered_dataset['user_id'].nunique()}")
print(f"  Number of items: {filtered_dataset['item_id'].nunique()}")
print(f"  Number of interactions: {len(filtered_dataset)}")
print(f"  Item interaction range: {filtered_dataset['item_id'].value_counts().min()} ~ {filtered_dataset['item_id'].value_counts().max()}")

# 3. Remap IDs (consistent with original code)
print("\n3. Remapping IDs...")
all_user = filtered_dataset.user_id
all_item = filtered_dataset.item_id

user_id, user_token = pd.factorize(all_user)
item_id, item_token = pd.factorize(all_item)

num_users = len(user_token) + 1  # 0 id is for PAD
num_items = len(item_token) + 1  # 0 id is for PAD

user_mapping_dict = {_: idx + 1 for idx, _ in enumerate(user_token)}  # 0 id is for PAD
item_mapping_dict = {_: idx + 1 for idx, _ in enumerate(item_token)}  # 0 id is for PAD

print(f"User mapping: {user_token.shape}")
print(f"Item mapping: {item_token.shape}")

filtered_dataset['user_id'] = filtered_dataset['user_id'].apply(lambda x: user_mapping_dict[x])
filtered_dataset['item_id'] = filtered_dataset['item_id'].apply(lambda x: item_mapping_dict[x])

# Save clean item list and counts
clean_items = sorted(filtered_dataset['item_id'].unique())
clean_item_counts = filtered_dataset['item_id'].value_counts().to_dict()
print(f"Clean item count: {len(clean_items)}")
print(f"Clean item ID range: {clean_items[0]} ~ {clean_items[-1]}")
print(f"Clean minimum interaction count: {min(clean_item_counts.values())}")

# 4. Create output directory
os.makedirs(output_path, exist_ok=True)

# 5. Process clean data
print(f"\n{'='*40}")
print(f"Processing clean data...")
print(f"{'='*40}")

clean_dataset = filtered_dataset.copy()
clean_type = 'clean'

# Create output directory
clean_dir = os.path.join(output_path, dataset_name, clean_type)
os.makedirs(clean_dir, exist_ok=True)

# Save inter.csv
csv_path = os.path.join(clean_dir, 'inter.csv')
clean_dataset.to_csv(csv_path, sep=',', index=None)
print(f"  Saving {clean_type} interaction data to: {csv_path}")

# Generate sequence data following original logic
print(f"  Generating sequence data...")

# Sort by user and time (original code logic)
clean_dataset_sorted = clean_dataset.sort_values(by=['user_id', 'timestamp'])

# ==================== Generate seq2pat_data.pth ====================
def to_list(x):
    return list(x)[:-2]  # Remove last 2 interactions, maintain original logic

user_group_for_seq2pat = clean_dataset_sorted.groupby('user_id')['item_id'].apply(to_list)

# Save seq2pat_data.pth
seq2pat_path = os.path.join(clean_dir, 'seq2pat_data.pth')
torch.save(user_group_for_seq2pat.tolist(), seq2pat_path)
print(f"  Generated {clean_type}/seq2pat_data.pth")
# ==================== seq2pat_data generation ends ====================

# Continue generating train/val/test sequences
user_group = clean_dataset_sorted.groupby('user_id')['item_id'].apply(list)

# Initialize lists
train, val, test = [], [], []

PAD = 0

# Process each user
for user_id, user_seq in list(zip(user_group.index, user_group.tolist())):
    # Truncate to maximum sequence length
    user_seq = user_seq[-max_seq_len:]
    
    # Skip users with sequences too short
    if len(user_seq) < 3:
        continue
        
    # ------ Test sample ------------
    history, seq_len = truncate_or_pad(user_seq[:-1])
    target_data = user_seq[-1]
    label = 1
    domain_id = [0] * max_seq_len  # domain fixed to 0
    test.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Validation sample -------------
    history, seq_len = truncate_or_pad(user_seq[:-2])
    target_data = user_seq[-2]
    label = 1
    domain_id = [0] * max_seq_len
    val.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Training sample -----------
    # Maintain original logic
    history, seq_len = truncate_or_pad(user_seq[:-3])
    target_data, _ = truncate_or_pad(user_seq[-seq_len-2:-2])
    label = [1] * seq_len + [PAD] * (max_seq_len - seq_len)
    domain_id = [0] * max_seq_len
    train.append([user_id, history, target_data, seq_len, label, domain_id])

# Save sequence data
torch.save(train, os.path.join(clean_dir, 'train.pth'))
torch.save(val, os.path.join(clean_dir, 'val.pth'))
torch.save(test, os.path.join(clean_dir, 'test.pth'))

print(f"  Saved sequence data: training={len(train)}, validation={len(val)}, test={len(test)}")

# 6. Generate 10% noise data (precise control)
print(f"\n{'='*40}")
print(f"Processing 10% noise data (precise noise ratio control)...")
print(f"{'='*40}")

# Generate noise data (using new precise control function)
noisy_dataset, total_changes, total_interactions = add_noise_guarantee_threshold(
    filtered_dataset.copy(), 
    noise_ratio=NOISE_RATIO
)

if noisy_dataset is None:
    print("Noise addition failed, cannot satisfy all conditions!")
    exit(1)

noisy_type = f'noisy_{int(NOISE_RATIO*100)}'

# Create output directory
noisy_dir = os.path.join(output_path, dataset_name, noisy_type)
os.makedirs(noisy_dir, exist_ok=True)

# Save inter.csv
noisy_csv_path = os.path.join(noisy_dir, 'inter.csv')
noisy_dataset.to_csv(noisy_csv_path, sep=',', index=None)
print(f"  Saving {noisy_type} interaction data to: {noisy_csv_path}")

# Generate sequence data following original logic
print(f"  Generating sequence data...")

# Sort by user and time (original code logic)
noisy_dataset_sorted = noisy_dataset.sort_values(by=['user_id', 'timestamp'])

# ==================== Generate seq2pat_data.pth ====================
user_group_for_seq2pat_noisy = noisy_dataset_sorted.groupby('user_id')['item_id'].apply(to_list)

# Save seq2pat_data.pth
seq2pat_path_noisy = os.path.join(noisy_dir, 'seq2pat_data.pth')
torch.save(user_group_for_seq2pat_noisy.tolist(), seq2pat_path_noisy)
print(f"  Generated {noisy_type}/seq2pat_data.pth")
# ==================== seq2pat_data generation ends ====================

# Continue generating train/val/test sequences
user_group_noisy = noisy_dataset_sorted.groupby('user_id')['item_id'].apply(list)

# Initialize lists
train_noisy, val_noisy, test_noisy = [], [], []

# Process each user
for user_id, user_seq in list(zip(user_group_noisy.index, user_group_noisy.tolist())):
    # Truncate to maximum sequence length
    user_seq = user_seq[-max_seq_len:]
    
    # Skip users with sequences too short
    if len(user_seq) < 3:
        continue
        
    # ------ Test sample ------------
    history, seq_len = truncate_or_pad(user_seq[:-1])
    target_data = user_seq[-1]
    label = 1
    domain_id = [0] * max_seq_len
    test_noisy.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Validation sample -------------
    history, seq_len = truncate_or_pad(user_seq[:-2])
    target_data = user_seq[-2]
    label = 1
    domain_id = [0] * max_seq_len
    val_noisy.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Training sample -----------
    history, seq_len = truncate_or_pad(user_seq[:-3])
    target_data, _ = truncate_or_pad(user_seq[-seq_len-2:-2])
    label = [1] * seq_len + [PAD] * (max_seq_len - seq_len)
    domain_id = [0] * max_seq_len
    train_noisy.append([user_id, history, target_data, seq_len, label, domain_id])

# Save sequence data
torch.save(train_noisy, os.path.join(noisy_dir, 'train.pth'))
torch.save(val_noisy, os.path.join(noisy_dir, 'val.pth'))
torch.save(test_noisy, os.path.join(noisy_dir, 'test.pth'))

print(f"  Saved sequence data: training={len(train_noisy)}, validation={len(val_noisy)}, test={len(test_noisy)}")

print("\n" + "="*60)
print("Data processing completed!")
print("="*60)

# 7. Final verification
print(f"\n{'='*40}")
print("Final verification results:")
print(f"{'='*40}")

# Verify item count consistency
df_clean = pd.read_csv(csv_path)
df_noisy = pd.read_csv(noisy_csv_path)

clean_items_set = set(df_clean['item_id'].unique())
noisy_items_set = set(df_noisy['item_id'].unique())

if clean_items_set == noisy_items_set:
    print(f"Item ID sets are identical: {len(clean_items_set)} items")
else:
    print(f"Item ID sets do not match!")
    print(f"   Clean unique: {clean_items_set - noisy_items_set}")
    print(f"   Noisy unique: {noisy_items_set - clean_items_set}")

# Verify interaction counts
noisy_item_counts = df_noisy['item_id'].value_counts()
clean_item_counts = df_clean['item_id'].value_counts()

if noisy_item_counts.min() >= item_threshold:
    print(f" All items have interaction count ≥ {item_threshold}")
    print(f"   Noisy minimum interactions: {noisy_item_counts.min()}")
    print(f"   Clean minimum interactions: {clean_item_counts.min()}")
else:
    print(f"Some items have interaction count < {item_threshold}")
    below_threshold = noisy_item_counts[noisy_item_counts < item_threshold]
    print(f"   Count: {len(below_threshold)}")

# Verify total interaction count
if len(df_clean) == len(df_noisy):
    print(f"Total interaction count consistent: {len(df_clean)}")
else:
    print(f"Total interaction count inconsistent: {len(df_clean)} vs {len(df_noisy)}")

# Verify user count
if df_clean['user_id'].nunique() == df_noisy['user_id'].nunique():
    print(f"User count consistent: {df_clean['user_id'].nunique()}")
else:
    print(f"User count inconsistent")

# Calculate actual noise ratio
actual_noise_ratio = total_changes / total_interactions if total_interactions > 0 else 0
print(f"\nNoise ratio statistics:")
print(f"  Target noise ratio: {NOISE_RATIO*100:.1f}%")
print(f"  Actual noise ratio: {actual_noise_ratio*100:.2f}%")
print(f"  Absolute error: {abs(actual_noise_ratio - NOISE_RATIO)*100:.3f}%")

print(f"\nOutput directory structure:")
print(f"{output_path}/")
print(f"└── {dataset_name}/")
print(f"    ├── clean/")
print(f"    │   ├── inter.csv")
print(f"    │   ├── seq2pat_data.pth")
print(f"    │   ├── train.pth")
print(f"    │   ├── val.pth")
print(f"    │   └── test.pth")
print(f"    └── noisy_10/")
print(f"        ├── inter.csv")
print(f"        ├── seq2pat_data.pth")
print(f"        ├── train.pth")
print(f"        ├── val.pth")
print(f"        └── test.pth")

print(f"\nKey guarantees:")
print(f"1. All items have interaction count ≥ {item_threshold}")
print(f"2. Item count unchanged ({len(clean_items_set)} items)")
print(f"3. No new items introduced, no items lost")
print(f"4. Precisely controlled noise ratio ≈ 10%")

print("\nExperiment ready!")

yelp Replacement

In [ ]:
import pandas as pd
import os
import random
import torch

# Set random seed
random.seed(2024)

# ==================== Configuration Parameters ====================
input_path = 'C:/Users/THINK BOOK-16/Desktop/yelp/'  # Directory containing inter.csv
output_path = 'C:/Users/THINK BOOK-16/Desktop/yelp-processed-15/'
dataset_name = 'yelp'

user_threshold = 5
item_threshold = 5
max_seq_len = 50

# ==================== Generate only 15% noisy version ====================
NOISE_RATIO = 0.15  # Generate only 15% noise

# ==================== Helper Functions ====================
def add_noise_guarantee_threshold(dataset, noise_ratio=0.05):
    """
    Add noise while ensuring:
    1. All items have interaction count ≥ item_threshold
    2. Item count remains unchanged (no new items introduced, no items lost)
    3. Only modify item_id
    4. Precisely control total noise ratio to noise_ratio
    """
    # Get clean data information
    clean_items = dataset['item_id'].unique().tolist()
    clean_item_counts = dataset['item_id'].value_counts().to_dict()
    
    print(f"  Clean item count: {len(clean_items)}")
    print(f"  Clean minimum interaction count: {min(clean_item_counts.values())}")
    
    # Copy dataset
    noisy_dataset = dataset.copy()
    
    # Calculate total interactions to modify
    total_interactions = len(noisy_dataset)
    target_changes = int(total_interactions * noise_ratio)
    print(f"  Total interactions: {total_interactions}")
    print(f"  Target modifications: {target_changes} (expected noise ratio: {noise_ratio*100:.1f}%)")
    
    # Track current interaction counts for each item (dynamically updated)
    current_item_counts = clean_item_counts.copy()
    
    # Phase 1: Collect all possible modifyable interactions
    print("  Phase 1: Collecting candidate modification positions...")
    candidate_indices = []
    
    for idx in noisy_dataset.index:
        old_item = noisy_dataset.at[idx, 'item_id']
        # Only consider items that would still meet threshold after decreasing by 1
        if current_item_counts[old_item] > item_threshold:
            candidate_indices.append(idx)
    
    print(f"  Found {len(candidate_indices)} candidate modification positions")
    
    # Randomly shuffle candidate positions
    random.shuffle(candidate_indices)
    
    # Phase 2: Perform modifications until target is reached
    print("  Phase 2: Executing modifications...")
    total_changes = 0
    max_attempts_per_change = 50
    
    for idx in candidate_indices:
        if total_changes >= target_changes:
            break
            
        old_item = noisy_dataset.at[idx, 'item_id']
        
        # Try to find a suitable replacement item
        attempts = 0
        new_item = None
        
        while attempts < max_attempts_per_change and new_item is None:
            # Randomly select a different item
            candidate = random.choice([item for item in clean_items if item != old_item])
            
            # Check if replacement is safe
            if (current_item_counts[old_item] - 1 >= item_threshold and 
                current_item_counts[candidate] + 1 >= item_threshold):
                new_item = candidate
                break
            attempts += 1
        
        if new_item is None:
            # Cannot find suitable replacement, skip
            continue
        
        # Perform replacement
        noisy_dataset.at[idx, 'item_id'] = new_item
        
        # Update counts
        current_item_counts[old_item] -= 1
        current_item_counts[new_item] += 1
        
        total_changes += 1
        
        # Show progress
        if total_changes % 500 == 0 or total_changes == target_changes:
            progress = total_changes / target_changes * 100
            print(f"    Progress: {total_changes}/{target_changes} ({progress:.1f}%)")
    
    # Calculate actual noise ratio
    actual_changes = 0
    for idx in noisy_dataset.index:
        # Find corresponding original index
        if idx in dataset.index:
            if dataset.at[idx, 'item_id'] != noisy_dataset.at[idx, 'item_id']:
                actual_changes += 1
    
    actual_noise_ratio = actual_changes / total_interactions
    
    print(f"\n  Noise addition completed:")
    print(f"    Target modifications: {target_changes}")
    print(f"    Actual modifications: {total_changes}")
    print(f"    Actual noise ratio: {actual_noise_ratio*100:.2f}%")
    
    # If actual ratio deviates significantly from target, show warning
    if abs(actual_noise_ratio - noise_ratio) > 0.005:  # 0.5% tolerance
        print(f"  Notice: Actual noise ratio deviates from target")
        print(f"    Target: {noise_ratio*100:.2f}%")
        print(f"    Actual: {actual_noise_ratio*100:.2f}%")
    
    # Verify item ID set
    noisy_items = set(noisy_dataset['item_id'].unique())
    clean_items_set = set(clean_items)
    
    if noisy_items == clean_items_set:
        print(f"  Item ID sets are identical")
    else:
        print(f"  Item ID sets do not match!")
        print(f"    Clean unique: {clean_items_set - noisy_items}")
        print(f"    Noisy unique: {noisy_items - clean_items_set}")
        return None, 0, 0
    
    # Verify interaction counts
    item_counts = noisy_dataset['item_id'].value_counts()
    min_interactions = item_counts.min()
    
    if min_interactions >= item_threshold:
        print(f"  All items have interaction count ≥ {item_threshold}")
        print(f"    Minimum interaction count: {min_interactions}")
    else:
        print(f"  Some items have interaction count < {item_threshold}")
        below_threshold = item_counts[item_counts < item_threshold]
        print(f"    Count: {len(below_threshold)}")
        print(f"    Item IDs: {list(below_threshold.index)}")
        return None, 0, 0
    
    # Verify total interaction count unchanged
    if len(noisy_dataset) == len(dataset):
        print(f"  Total interaction count unchanged: {len(noisy_dataset)}")
    else:
        print(f"  Total interaction count changed: {len(dataset)} -> {len(noisy_dataset)}")
        return None, 0, 0
    
    return noisy_dataset, total_changes, total_interactions

def truncate_or_pad(seq):
    """Truncate or pad sequence to fixed length (maintain original logic)"""
    cur_seq_len = len(seq)
    if cur_seq_len > max_seq_len:
        return seq[-max_seq_len:], max_seq_len
    else:
        PAD = 0
        return seq + [PAD] * (max_seq_len - cur_seq_len), cur_seq_len

# ==================== Main Process ====================

print("="*60)
print("Starting Yelp Dataset Processing (15% noise addition, precise noise ratio control)")
print("="*60)

# 1. Load inter.csv file
print("\n1. Loading inter.csv file...")
inter_file = os.path.join(input_path, 'inter.csv')

if not os.path.exists(inter_file):
    print(f"Error: Cannot find file {inter_file}")
    exit(1)

# Read inter.csv file
dataset = pd.read_csv(inter_file)

print(f"Original data statistics:")
print(f"  Data shape: {dataset.shape}")
print(f"  Number of users: {dataset['user_id'].nunique()}")
print(f"  Number of items: {dataset['item_id'].nunique()}")
print(f"  Number of interactions: {len(dataset)}")

# Check if domain column exists, if not add one
if 'domain' not in dataset.columns:
    print("Note: No domain column in inter.csv, adding default domain=0")
    dataset['domain'] = 0

# 2. Filter dataset (based on interaction frequency)
print("\n2. Filtering dataset...")
filtered_dataset = dataset.copy()
while True:
    ori_len = len(filtered_dataset)
    
    # Filter users with less than user_threshold interactions
    filtered_dataset = filtered_dataset[filtered_dataset['user_id'].map(filtered_dataset['user_id'].value_counts()) >= user_threshold]
    # Filter items with less than item_threshold interactions
    filtered_dataset = filtered_dataset[filtered_dataset['item_id'].map(filtered_dataset['item_id'].value_counts()) >= item_threshold]
    
    if len(filtered_dataset) == ori_len:
        break

print(f"\nFiltered data statistics:")
print(f"  Number of users: {filtered_dataset['user_id'].nunique()}")
print(f"  Number of items: {filtered_dataset['item_id'].nunique()}")
print(f"  Number of interactions: {len(filtered_dataset)}")
print(f"  Item interaction range: {filtered_dataset['item_id'].value_counts().min()} ~ {filtered_dataset['item_id'].value_counts().max()}")

# 3. Remap IDs (consistent with original code)
print("\n3. Remapping IDs...")
all_user = filtered_dataset.user_id
all_item = filtered_dataset.item_id

user_id, user_token = pd.factorize(all_user)
item_id, item_token = pd.factorize(all_item)

num_users = len(user_token) + 1  # 0 id is for PAD
num_items = len(item_token) + 1  # 0 id is for PAD

user_mapping_dict = {_: idx + 1 for idx, _ in enumerate(user_token)}  # 0 id is for PAD
item_mapping_dict = {_: idx + 1 for idx, _ in enumerate(item_token)}  # 0 id is for PAD

print(f"User mapping: {user_token.shape}")
print(f"Item mapping: {item_token.shape}")

filtered_dataset['user_id'] = filtered_dataset['user_id'].apply(lambda x: user_mapping_dict[x])
filtered_dataset['item_id'] = filtered_dataset['item_id'].apply(lambda x: item_mapping_dict[x])

# Save clean item list and counts
clean_items = sorted(filtered_dataset['item_id'].unique())
clean_item_counts = filtered_dataset['item_id'].value_counts().to_dict()
print(f"Clean item count: {len(clean_items)}")
print(f"Clean item ID range: {clean_items[0]} ~ {clean_items[-1]}")
print(f"Clean minimum interaction count: {min(clean_item_counts.values())}")

# 4. Create output directory
os.makedirs(output_path, exist_ok=True)

# 5. Process clean data
print(f"\n{'='*40}")
print(f"Processing clean data...")
print(f"{'='*40}")

clean_dataset = filtered_dataset.copy()
clean_type = 'clean'

# Create output directory
clean_dir = os.path.join(output_path, dataset_name, clean_type)
os.makedirs(clean_dir, exist_ok=True)

# Save inter.csv
csv_path = os.path.join(clean_dir, 'inter.csv')
clean_dataset.to_csv(csv_path, sep=',', index=None)
print(f"  Saving {clean_type} interaction data to: {csv_path}")

# Generate sequence data following original logic
print(f"  Generating sequence data...")

# Sort by user and time (original code logic)
clean_dataset_sorted = clean_dataset.sort_values(by=['user_id', 'timestamp'])

# ==================== Generate seq2pat_data.pth ====================
def to_list(x):
    return list(x)[:-2]  # Remove last 2 interactions, maintain original logic

user_group_for_seq2pat = clean_dataset_sorted.groupby('user_id')['item_id'].apply(to_list)

# Save seq2pat_data.pth
seq2pat_path = os.path.join(clean_dir, 'seq2pat_data.pth')
torch.save(user_group_for_seq2pat.tolist(), seq2pat_path)
print(f"  Generated {clean_type}/seq2pat_data.pth")
# ==================== seq2pat_data generation ends ====================

# Continue generating train/val/test sequences
user_group = clean_dataset_sorted.groupby('user_id')['item_id'].apply(list)

# Initialize lists
train, val, test = [], [], []

PAD = 0

# Process each user
for user_id, user_seq in list(zip(user_group.index, user_group.tolist())):
    # Truncate to maximum sequence length
    user_seq = user_seq[-max_seq_len:]
    
    # Skip users with sequences too short
    if len(user_seq) < 3:
        continue
        
    # ------ Test sample ------------
    history, seq_len = truncate_or_pad(user_seq[:-1])
    target_data = user_seq[-1]
    label = 1
    domain_id = [0] * max_seq_len  # domain fixed to 0 (consistent with Amazon)
    test.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Validation sample -------------
    history, seq_len = truncate_or_pad(user_seq[:-2])
    target_data = user_seq[-2]
    label = 1
    domain_id = [0] * max_seq_len
    val.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Training sample -----------
    # Maintain original logic (consistent with Amazon)
    history, seq_len = truncate_or_pad(user_seq[:-3])
    target_data, _ = truncate_or_pad(user_seq[-seq_len-2:-2])
    label = [1] * seq_len + [PAD] * (max_seq_len - seq_len)
    domain_id = [0] * max_seq_len
    train.append([user_id, history, target_data, seq_len, label, domain_id])

# Save sequence data
torch.save(train, os.path.join(clean_dir, 'train.pth'))
torch.save(val, os.path.join(clean_dir, 'val.pth'))
torch.save(test, os.path.join(clean_dir, 'test.pth'))

print(f"  Saved sequence data: training={len(train)}, validation={len(val)}, test={len(test)}")

# 6. Generate 15% noise data (precise control)
print(f"\n{'='*40}")
print(f"Processing 15% noise data (precise noise ratio control)...")
print(f"{'='*40}")

# Generate noise data (using new precise control function)
noisy_dataset, total_changes, total_interactions = add_noise_guarantee_threshold(
    filtered_dataset.copy(), 
    noise_ratio=NOISE_RATIO
)

if noisy_dataset is None:
    print("Noise addition failed, cannot satisfy all conditions!")
    exit(1)

noisy_type = f'noisy_{int(NOISE_RATIO*100)}'

# Create output directory
noisy_dir = os.path.join(output_path, dataset_name, noisy_type)
os.makedirs(noisy_dir, exist_ok=True)

# Save inter.csv
noisy_csv_path = os.path.join(noisy_dir, 'inter.csv')
noisy_dataset.to_csv(noisy_csv_path, sep=',', index=None)
print(f"  Saving {noisy_type} interaction data to: {noisy_csv_path}")

# Generate sequence data following original logic
print(f"  Generating sequence data...")

# Sort by user and time (original code logic)
noisy_dataset_sorted = noisy_dataset.sort_values(by=['user_id', 'timestamp'])

# ==================== Generate seq2pat_data.pth ====================
user_group_for_seq2pat_noisy = noisy_dataset_sorted.groupby('user_id')['item_id'].apply(to_list)

# Filter out empty lists
user_group_for_seq2pat_noisy = [seq for seq in user_group_for_seq2pat.tolist() if len(seq) > 0]

# Save seq2pat_data.pth
seq2pat_path_noisy = os.path.join(noisy_dir, 'seq2pat_data.pth')
torch.save(user_group_for_seq2pat_noisy, seq2pat_path_noisy)
print(f"  Generated {noisy_type}/seq2pat_data.pth")
# ==================== seq2pat_data generation ends ====================

# Continue generating train/val/test sequences
user_group_noisy = noisy_dataset_sorted.groupby('user_id')['item_id'].apply(list)

# Initialize lists
train_noisy, val_noisy, test_noisy = [], [], []

# Process each user
for user_id, user_seq in list(zip(user_group_noisy.index, user_group_noisy.tolist())):
    # Truncate to maximum sequence length
    user_seq = user_seq[-max_seq_len:]
    
    # Skip users with sequences too short
    if len(user_seq) < 3:
        continue
        
    # ------ Test sample ------------
    history, seq_len = truncate_or_pad(user_seq[:-1])
    target_data = user_seq[-1]
    label = 1
    domain_id = [0] * max_seq_len
    test_noisy.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Validation sample -------------
    history, seq_len = truncate_or_pad(user_seq[:-2])
    target_data = user_seq[-2]
    label = 1
    domain_id = [0] * max_seq_len
    val_noisy.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Training sample -----------
    history, seq_len = truncate_or_pad(user_seq[:-3])
    target_data, _ = truncate_or_pad(user_seq[-seq_len-2:-2])
    label = [1] * seq_len + [PAD] * (max_seq_len - seq_len)
    domain_id = [0] * max_seq_len
    train_noisy.append([user_id, history, target_data, seq_len, label, domain_id])

# Save sequence data
torch.save(train_noisy, os.path.join(noisy_dir, 'train.pth'))
torch.save(val_noisy, os.path.join(noisy_dir, 'val.pth'))
torch.save(test_noisy, os.path.join(noisy_dir, 'test.pth'))

print(f"  Saved sequence data: training={len(train_noisy)}, validation={len(val_noisy)}, test={len(test_noisy)}")

print("\n" + "="*60)
print("Data processing completed!")
print("="*60)

# 7. Final verification
print(f"\n{'='*40}")
print("Final verification results:")
print(f"{'='*40}")

# Verify item count consistency
df_clean = pd.read_csv(csv_path)
df_noisy = pd.read_csv(noisy_csv_path)

clean_items_set = set(df_clean['item_id'].unique())
noisy_items_set = set(df_noisy['item_id'].unique())

if clean_items_set == noisy_items_set:
    print(f"Item ID sets are identical: {len(clean_items_set)} items")
else:
    print(f"Item ID sets do not match!")
    print(f"   Clean unique: {clean_items_set - noisy_items_set}")
    print(f"   Noisy unique: {noisy_items_set - clean_items_set}")

# Verify interaction counts
noisy_item_counts = df_noisy['item_id'].value_counts()
clean_item_counts = df_clean['item_id'].value_counts()

if noisy_item_counts.min() >= item_threshold:
    print(f"All items have interaction count ≥ {item_threshold}")
    print(f"   Noisy minimum interactions: {noisy_item_counts.min()}")
    print(f"   Clean minimum interactions: {clean_item_counts.min()}")
else:
    print(f"Some items have interaction count < {item_threshold}")
    below_threshold = noisy_item_counts[noisy_item_counts < item_threshold]
    print(f"   Count: {len(below_threshold)}")

# Verify total interaction count
if len(df_clean) == len(df_noisy):
    print(f"Total interaction count consistent: {len(df_clean)}")
else:
    print(f"Total interaction count inconsistent: {len(df_clean)} vs {len(df_noisy)}")

# Verify user count
if df_clean['user_id'].nunique() == df_noisy['user_id'].nunique():
    print(f"User count consistent: {df_clean['user_id'].nunique()}")
else:
    print(f"User count inconsistent")

# Calculate actual noise ratio
actual_noise_ratio = total_changes / total_interactions if total_interactions > 0 else 0
print(f"\nNoise ratio statistics:")
print(f"  Target noise ratio: {NOISE_RATIO*100:.1f}%")
print(f"  Actual noise ratio: {actual_noise_ratio*100:.2f}%")
print(f"  Absolute error: {abs(actual_noise_ratio - NOISE_RATIO)*100:.3f}%")

print(f"\nOutput directory structure:")
print(f"{output_path}/")
print(f"└── {dataset_name}/")
print(f"    ├── clean/")
print(f"    │   ├── inter.csv")
print(f"    │   ├── seq2pat_data.pth")
print(f"    │   ├── train.pth")
print(f"    │   ├── val.pth")
print(f"    │   └── test.pth")
print(f"    └── noisy_15/")
print(f"        ├── inter.csv")
print(f"        ├── seq2pat_data.pth")
print(f"        ├── train.pth")
print(f"        ├── val.pth")
print(f"        └── test.pth")

print(f"\nKey guarantees:")
print(f"1. All items have interaction count ≥ {item_threshold}")
print(f"2. Item count unchanged ({len(clean_items_set)} items)")
print(f"3. No new items introduced, no items lost")
print(f"4. Precisely controlled noise ratio ≈ 15%")

print("\nYelp experiment ready!")